# 00 - Setup and synthetic raw data

Creates the Unity Catalog objects and writes three raw CSV files to a UC Volume:

- `positions.csv`: trading-book positions with their sensitivity to one risk factor
- `scenarios.csv`: stress scenarios as shocks per risk factor
- `limits.csv`: stress loss limit per desk

All data is synthetic (fixed random seed). Two invalid rows are added on purpose
so the pipeline's data quality rules have something to catch.

In [1]:
import csv, io, random, yaml
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient

import logging, warnings
warnings.filterwarnings("ignore")
logging.getLogger("databricks.sdk").setLevel(logging.ERROR)

cfg = yaml.safe_load(open("../config.yml"))
spark = DatabricksSession.builder.profile(cfg["profile"]).serverless(True).getOrCreate()
w = WorkspaceClient(profile=cfg["profile"])
cat, sch, vol = cfg["catalog"], cfg["schema"], cfg["volume"]
raw = f"/Volumes/{cat}/{sch}/{vol}"

## Unity Catalog objects
The catalog itself is created once beforehand (this metastore needs a managed location, see README).
The column mask is created here because the pipeline attaches it to `silver_positions`.
Only members of `risk_admins` see counterparty names; everyone else sees `REDACTED`.

In [2]:
for stmt in [
    f"CREATE SCHEMA IF NOT EXISTS {cat}.{sch}",
    f"CREATE VOLUME IF NOT EXISTS {cat}.{sch}.{vol} COMMENT 'Raw CSV landing zone'",
    f"""CREATE OR REPLACE FUNCTION {cat}.{sch}.mask_counterparty(name STRING)
        RETURN CASE WHEN is_account_group_member('risk_admins') THEN name ELSE 'REDACTED' END""",
]:
    spark.sql(stmt)
spark.sql(f"SHOW VOLUMES IN {cat}.{sch}").show()

+--------+-----------+
|database|volume_name|
+--------+-----------+
|    risk|        raw|
+--------+-----------+



## Synthetic positions
`sensitivity_jpy` is the P&L in JPY for a +1 unit move of the risk factor:
1 bp for rates and credit spreads, 1 % for FX and equity.

In [3]:
random.seed(42)
FACTORS = {
    "Rates": ("IRS", ["JPY_IR_10Y", "USD_IR_10Y"]),
    "FX": ("FX_FORWARD", ["USDJPY", "EURJPY"]),
    "Equity": ("EQUITY_FUTURE", ["NIKKEI225", "SPX"]),
    "Credit": ("CORPORATE_BOND", ["JP_CREDIT_A", "US_CREDIT_BBB"]),
}

positions = []
for i in range(1, 201):
    desk = random.choice(list(FACTORS))
    instrument, factors = FACTORS[desk]
    notional = random.randint(1, 50) * 100_000_000
    long = random.random() < 0.7  # like a typical bank book, most positions are long the market
    if desk in ("Rates", "Credit"):  # long bonds lose when rates or spreads rise
        sensitivity = (-1 if long else 1) * notional * random.uniform(1, 10) * 1e-4  # DV01 / CS01
    else:  # long USD and long equity lose when they fall
        sensitivity = (1 if long else -1) * notional * 0.01  # delta per 1 %
    positions.append([f"P{i:04d}", desk, instrument, random.choice(factors),
                      round(sensitivity), notional, f"Counterparty {random.randint(1, 30):03d}"])

positions += [  # invalid rows for the data quality rules
    ["P9001", "Unknown", "IRS", "JPY_IR_10Y", 1000, 100_000_000, "Counterparty 001"],
    ["P9002", "Rates", "IRS", "JPY_IR_10Y", "", 100_000_000, "Counterparty 002"],
]

## Stress scenarios
Shocks are in bp for rates and credit spreads, and in % for FX and equity.

In [4]:
SCENARIOS = {
    "S1": ("BoJ hikes 100bp", {"JPY_IR_10Y": 100, "USD_IR_10Y": 20, "USDJPY": -5, "EURJPY": -4,
                                "NIKKEI225": -8, "SPX": -2, "JP_CREDIT_A": 15, "US_CREDIT_BBB": 5}),
    "S2": ("Yen surges 15%", {"JPY_IR_10Y": -10, "USD_IR_10Y": -15, "USDJPY": -15, "EURJPY": -14,
                               "NIKKEI225": -12, "SPX": -3, "JP_CREDIT_A": 10, "US_CREDIT_BBB": 20}),
    "S3": ("Global equity sell-off", {"JPY_IR_10Y": -15, "USD_IR_10Y": -50, "USDJPY": -8, "EURJPY": -9,
                                       "NIKKEI225": -25, "SPX": -20, "JP_CREDIT_A": 40, "US_CREDIT_BBB": 120}),
    "S4": ("Credit spreads widen", {"JPY_IR_10Y": -5, "USD_IR_10Y": -30, "USDJPY": -3, "EURJPY": -3,
                                     "NIKKEI225": -10, "SPX": -8, "JP_CREDIT_A": 60, "US_CREDIT_BBB": 200}),
}
scenarios = [[sid, name, rf, shock] for sid, (name, shocks) in SCENARIOS.items() for rf, shock in shocks.items()]

In [5]:
def upload(path, header, rows):
    buf = io.StringIO()
    csv.writer(buf).writerows([header] + rows)
    w.files.upload(path, io.BytesIO(buf.getvalue().encode()), overwrite=True)
    print(f"wrote {len(rows)} rows to {path}")

upload(f"{raw}/positions/positions.csv",
       ["position_id", "desk", "instrument_type", "risk_factor", "sensitivity_jpy", "notional_jpy", "counterparty"],
       positions)
upload(f"{raw}/scenarios/scenarios.csv", ["scenario_id", "scenario_name", "risk_factor", "shock"], scenarios)

# Stress loss limit per desk (JPY): the loss the board accepts under any single scenario.
limits = [["Rates", 2_000_000_000], ["FX", 8_000_000_000], ["Equity", 5_000_000_000], ["Credit", 4_000_000_000]]
upload(f"{raw}/limits/limits.csv", ["desk", "stress_loss_limit_jpy"], limits)

wrote 202 rows to /Volumes/stress_demo/risk/raw/positions/positions.csv


wrote 32 rows to /Volumes/stress_demo/risk/raw/scenarios/scenarios.csv


wrote 4 rows to /Volumes/stress_demo/risk/raw/limits/limits.csv


## Check what landed in the Volume

In [6]:
spark.sql(f"""
  SELECT desk, count(*) AS positions
  FROM read_files('{raw}/positions/', format => 'csv', header => true)
  GROUP BY desk ORDER BY desk
""").show()
spark.sql(f"SELECT * FROM read_files('{raw}/scenarios/', format => 'csv', header => true) WHERE scenario_id = 'S1'").show()

+-------+---------+
|   desk|positions|
+-------+---------+
| Credit|       57|
| Equity|       50|
|     FX|       52|
|  Rates|       42|
|Unknown|        1|
+-------+---------+



+-----------+---------------+-------------+-----+-------------+
|scenario_id|  scenario_name|  risk_factor|shock|_rescued_data|
+-----------+---------------+-------------+-----+-------------+
|         S1|BoJ hikes 100bp|   JPY_IR_10Y|  100|         NULL|
|         S1|BoJ hikes 100bp|   USD_IR_10Y|   20|         NULL|
|         S1|BoJ hikes 100bp|       USDJPY|   -5|         NULL|
|         S1|BoJ hikes 100bp|       EURJPY|   -4|         NULL|
|         S1|BoJ hikes 100bp|    NIKKEI225|   -8|         NULL|
|         S1|BoJ hikes 100bp|          SPX|   -2|         NULL|
|         S1|BoJ hikes 100bp|  JP_CREDIT_A|   15|         NULL|
|         S1|BoJ hikes 100bp|US_CREDIT_BBB|    5|         NULL|
+-----------+---------------+-------------+-----+-------------+

